In [1]:

# Import pipy modules
from ipyfilechooser import FileChooser
from IPython import display
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# from rich import inspect, console
import traitlets
import xarray as xr

# Welcome to CIBUSmod_soil
This noteboook is an interactive interface meant to be used to perform the necessary calculations to determine the carbon stock changes resulting from a CIBUSmod scenario, as well as enable the user to do invididual updates to specific regions in a scenario and recalculate the values resulting of these partial changes to the food system.

Before running the commands the modules used in this notebook need to be imported. They are located in the cell below.

All other necessary modules are imported when required from the module containing the functions called

In [1]:
# Import builtin modules
import sys
import os
import importlib as il
sys.path.insert(0, os.path.join(os.getcwd(),'..'))
os.chdir('..')

In [2]:
# Import locally defined modules
import CIBUSmod as cm
from CIBUSmod.soil.soils import SoilData
from CIBUSmod.soil.soil_utils import colored_rule

root: /home/niceri/Pythoncode/CIBUSmod
input_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/input
temp_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
export_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/exported_results


In [3]:
import pandas as pd

# Print green bar on success
colored_rule(color='green', height=2)

In [4]:
# To create an input_df for the SoilData class from a session instance run:
# to_icbm(session) with arbitrary name

# To get the scenario name for the SoilData class from a session instance use:
# scenario_name = list(session.scenarios[0])
#
# If there are more than one scenario present in the current session 
# they can be extracted iteratively to run all operations on 
# one scenario at the time.
# Use the following code to collect the scenario names in a list: 
# scenario_names = []
# for i in list(session.scenarios)
#     scenario_names.append(i)

# For testing purposes the database has been exported from session as a csv
# and the scenario_name is defined as:
scenario_name = 'FAI'
#icbm_in = pd.read_csv('/home/niceri/Pythoncode/CIBUSmod-main/notebooks/FAI_to_ICBM.csv')
icbm_in = pd.read_csv('/home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input/FAI.csv')

new_soil = SoilData(icbm_in, scenario_name)

In [5]:
pwd()

'/home/niceri/Pythoncode/CIBUSmod'

In [6]:
colored_rule(color='magenta', height=2)
new_soil.calc_scn_inputs(verbose=True)
colored_rule(color='green', height=2)

---Executing calc_scn_inputs()---
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)
>>> Executing '_calc_input_ha()'<<<
>>> '_calc_input_ha()' executed succesfully <<<
>>> Executing '_calc_amnd_ha()'<<<
--- Inserted manure_cattle_ha---
--- Inserted manure_horses_ha---
--- Inserted manure_pigs_ha---
--- Inserted manure_poultry_ha---
>>> '_calc_amnd_ha()' executed succesfully <<<
>>> Executing '_calc_crop_inputs()'<<<
---Executing alloc_helper()---
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)
---Leaving alloc_helper()---
---Executing calculate_c_inputs()---
---Leaving calculate_c_inputs()---
>>> '_calc_crop_inputs()' executed succesfully <<<
>>> Executing '_add_prefixes()' <<<
>>> '_add_prefixes()' executed succesfully <<<
>>> Executing '_change_year_na

In [7]:
colored_rule(color='magenta', height=2)
new_soil.calc_soc_timeseries(verbose=True)
colored_rule(color='green', height=2)

---Executing _calculate_soc()---
---Executing _make_scn_area_dfs()---
---Leaving _make_scn_area_dfs()---
'_c_input_ha_df' and '_c_input_sko_df' generated
-> 'h_value_dict' not set.
---Executing h_map_helper()---
An h-value mapping dataframe does not exist.
Creating h_map_df from 'h_values.csv' in /home/niceri/Pythoncode/CIBUSmod/data/soil/input
CIBUS crop mapping dataframe does not exist
> Calling 'crop_map_helper()'
---Executing crop_map_helper()---
No input dataframe exists.
Creating 'input_df' from 'crop_carbon_map.csv'in /home/niceri/Pythoncode/CIBUSmod/data/soil/input
---Leaving crop_map_helper()---
'crop_in_df' and 'crop_re_dict' created
An amendment mapping dataframe does not exist
Creating 'amnd_map_df' from 'amnd_map.csv' in /home/niceri/Pythoncode/CIBUSmod/data/soil/input
'h_value_df' and 'h_value_dict' created
---Leaving h_map_helper()---
Extracting filtered_namelist per ha
Extracting filtered_namelist per sko
---Leaving _calculate_soc()---
---Executing _scn_icbm_calculation

In [10]:
new_soil.save_inventory('historic')

Historic SOC dataset saved as historic_soc_ds.nc 
 in /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results


In [9]:
colored_rule(color='magenta', height=2)
new_soil.calc_historic_soc_timeseries(verbose=True)
colored_rule(color='green', height=2)

---Executing _calculate_historic_soc()---
> One or both of '_ss_input_ha_df' and '_ss_input_sko_df' are unset
info: Generating 'spinup_ha_df' and 'spinup_sko_df' from 'input_df'
---Executing _make_spinup_area_dfs()---
---Leaving _make_spinup_area_dfs()---
'_ss_input_ha_df' and '_ss_input_sko_df' set using input_df for FAI
Extracting filtered_namelist per ha
Extracting filtered_namelist per sko
---Leaving _calculate_historic_soc()---
---Executing _historic_icbm_calculations()---
info: Calculating SOC SS values per ha in 2020
info: Calculating SOC SS values per sko in 2020
info: Calculating SOC timeseries per ha
info: Calculating SOC timeseries per sko
info: Finished calculating historic SOC timeseries
Creating xarray historic soc dataset
---_historic_icbm_calculations() executed succesfully---


# This is the interactive "Old" version of the noteboook
# 1. Calculate the carbon allocted to different fractions of each respective crop

<span style='color:orange'>Simplest way to use CIBUSmod_soil is to start by generating scenario_dfs and a spinup_df.</span>

After having generated the scenario and spinup dataframes the soc timeseries can be calculated (section 2) and exported to xarray dataseries. Scenario variable names should be changed to reflect the scenario names.

<span style='color:red'>Plotting and analysis of result is easier and better to do in xarray</span>

---

Run `make_scn_multi_df(scenario_name=<name of scenario>)`, where the `<name_of_scenario>` is the name of the `*.csv input file`

Run `make_spinup_df(input_df=<name_of_output_from_previous_function>)`. 

If no input df is given `make_scn_multi_df` and `make_scn_input_df`' will be called, recursively, with default values. That may take some time.

Supplying a ready made scenario dataframe with the `scn_inputs_df`' keyword is recommended.

---

* <span style='color:green'>All functions can be run without arguments. They then defaults to preset csv-files, which represent the FAI-scenario.</span>
* <span style='color:green'>Depending on which function is called, there are several optional keywords.</span>

  
<span style='color:red'>
Read the docstrings: `make_scn_input_df?`, `make_scn_multi_df?`, `make_spinup_df?` for detailed info. Add another `?` for the full source code.
</span>

---

<strong>continue to section 2</strong>

---
### To use intermediate functions independently read instructions below 

`make_scn_input_df()` creates a CIBUSmod scenario dataframe based on input data and allocation parameters.

This function takes input data for a CIBUSmod scenario, calculates yield and carbon input per unit area, and inserts new columns with manure input per hectare based on the total input per scenario. `alloc_helper() creates dictionaries to map cibus crop names to allocation factors or allometric functions

--------------------------------------------------------------------------------
Different sources and data values are mapped using df's created with the function below. These can be changed executing 'run make_df' and assigning the resulting df to a new df_variable to be used as input_df.

The df can be retrieved using `df_dict[list(df_dict)[0]]`

Sources used, in order of presedence, if deafult settings are used:

1. Andren 2004 (allometric functions)
2. Jacobs 2020 (allocation factors)
3. Hanna (allocation factors)

In [5]:
import CIBUSmod.soil.soil_utils as soil_utils
import CIBUSmod.soil.data_processing as data_processing

In [6]:
# Execute this cell to choose a scenario file
fc = soil_utils.select_file()
fc.title = '<b>Choose scenario file</b>'
display.display(fc)

FileChooser(path='/home/niceri/Pythoncode/CIBUSmod_soil/data/soil', filename='', title='<b>Choose scenario fil…

In [7]:
# Execute this cell to set the input_file and save scenario name to dict
input_file = fc.selected
scn_file_name = os.path.splitext(fc.selected_filename)[0]
input_name = input(f'Please enter the name of the scenario as it should appear in file names: [default: {scn_file_name}]')
scenario_dict = {'scenario_name': soil_utils.set_scn_name(input_name, scn_file_name), 'scenario_file': input_file}

Please enter the name of the scenario as it should appear in file names: [default: FAI] 


In [8]:
# Execute this cell to create scenario and spinup data frames
soil_utils.colored_rule(color='red', height=2)
scenario_input_df, scenario_input_ds, startyear = data_processing.make_scn_multi_df(scenario_dict=scenario_dict, verbose=True)
soil_utils.colored_rule(color='green', height=2)

---Executing make_scn_multi_df()---
scenario_name and scenario_file set to ('FAI', '/home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input/FAI.csv')
> scn_inputs_df not set
Calling make_scn_inputs_df()
---Executing make_scn_input_df()---
scenario name and scenario file is: ('FAI', '/home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input/FAI.csv') in make_scn_input_df
> Scenario file set
Creating scenario_input dataframe from /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input/FAI.csv
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)
Scenario_input dataframe created from /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input/FAI.csv
---Executing alloc_helper()---
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)
*** The following dict keys and their values ha

In [9]:
soil_utils.colored_rule(color='red', height=2)
spinup_df, endyear = data_processing.make_spinup_df(scenario_dict=scenario_dict, verbose=True)
soil_utils.colored_rule(color='green', height=2)

---Executing _make_spinup_df()---
> Input df not set
> Calling make_scn_multi_df()
---Executing make_scn_multi_df()---
scenario_name and scenario_file set to ('FAI', '/home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input/FAI.csv')
> scn_inputs_df not set
Setting 'scn_inputs_df' using 'scenario_dict'
_make_spinup_df called recursively. No scenario files saved
*** The following dict keys and their values have been added to 'scenario_dict ***
   'scenario_multi_df': Multiindex dataframe for scenario FAI.
   'scenario_input_ds': Xarray dataset for input in scenario FAI.
---Leaving make_scn_multi_df()---
>make_scn_multi_df completed<
Spinup dataframe saved to 'spinup_indput_df.csv' in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results
*** The following dict keys and their values have been added to 'scenario_dict ***
   'spinup_multi_df': Multiindex dataframe for spinup modelling.
---Leaving _make_spinup_df()---


# 2. Calculate the soc timeseries and export to xarray datasets

After having generated scenario and spinup dataframes the SOC timeseries can be calculated and exported to xarray by executing the functions below. Scenario variable names should be changed to reflect the scenario names

---

`calculate_scn_soc()` and `_calculate_historic_soc()` take dataframes with C inputs as input. They can be run with the keyword `indput_df=` using the `<scenario_input_df>`  and `<spinup_df>`  generated in step 1 (above). 

The functions return two dataframes: the first with values expressed per ha; the second with values expressed per sko (skördeområde).

They can also be run with two dataframes, which both are used independently to calculate soc timeseries, independently.

Before running the commands, change the `<variables>` to correct and meaningful names

---

In [25]:
il.reload(data_processing)
soil_utils.colored_rule(color='red', height=2)
scn_ha_soc_df, scn_sko_soc_df, scn_ha_soc_ds, scn_sko_soc_ds = data_processing.calculate_scn_soc(scenario_dict=scenario_dict, verbose=True)
soil_utils.colored_rule(color='green', height=2)

---Executing _calculate_soc()---
> One or both of 'scn_ha_df' and 'scn_sko_df' are not set
>_make_scn_area_dfs() called<
---Executing _make_scn_area_dfs()---
*** The following dict keys and their values have been added to 'scenario_dict ***
   'scenario_ha_input_df': Scenario FAI input dataframe expressed per ha.
   'scenario_sko_input_df': Scenario FAI input dataframe expressed per sko.
---Leaving _make_scn_area_dfs()---
'scn_ha_df' and 'scn_sko_df' set using input_df for FAI
>make_sc_area_dfs() completed
> 'h_value_dict' not set.
> Calling 'h_map_helper()'
---Executing h_map_helper()---
An h-value mapping dataframe does not exist.
Creating h_map_df from 'h_values.csv' in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input
CIBUS crop mapping dataframe does not exist
> Calling 'crop_map_helper()'
---Executing crop_map_helper()---
No input dataframe exists.
Creating 'input_df' from 'crop_carbon_map.csv'in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input
*** The following dict key

/home/niceri/Pythoncode/CIBUSmod_soil/notebooks/../CIBUSmod/soil/data_processing.py:624: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  scn_sko_prodsys_sum_df = input_df.groupby(scn_multi_groupby_idx).sum()


Calculating SOC timeseries per sko
Done calculating SOC timeseries
Creating xarray datasets and saving as netcdf-files
FAI-scenario SOC dataset saved as FAI_ha_soc_ds.nc 
 in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results
FAI-scenario SOC dataset saved as FAI_sko_soc_ds.nc 
 in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results
*** The following dict keys and their values have been added to 'scenario_dict ***
   'scenario_ha_soc_df': SOC dataframe expressed per ha for scenario FAI.
   'scenario_sko_soc_df': SOC dataframe expressed per sko for scenario FAI.
   'scenario_ha_soc_ds': xarray dataset of 'scenario_ha_soc_df'.
   'scenario_sko_soc_ds': xarray dataset of 'scenario_sko_soc_df'.
---Leaving _calculate_soc()---


In [7]:
#scn_ha_soc_ds = xr.load_dataset('intermediate_results/FAI_ha_soc_ds.nc')
scn_ha_soc_ds = xr.load_dataset('/home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results/FAI_soc_ds.nc')
historic_ha_soc_ds = xr.load_dataset('/home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results/')

FileNotFoundError: [Errno 2] No such file or directory: '/home/niceri/Pythoncode/CIBUSmod_soil/intermediate_results/FAI_ha_soc_ds.nc'

In [19]:
il.reload(data_processing)
soil_utils.colored_rule(color='red', height=2)
spinup_ha_soc_df, spinup_sko_soc_df, spinup_ha_soc_ds, spinup_sko_soc_ds = data_processing._calculate_historic_soc(input_df=spinup_df, scenario_dict=scenario_dict, verbose=True)
soil_utils.colored_rule(color='green', height=2)

---Executing _calculate_historic_soc()---
> One or both of 'spinup_ha_df' and 'spinup_sko_df' are not set
Generating 'spinup_ha_df' and 'spinup_sko_df' from 'input_df'
>'_make_spinup_area_dfs()' called<
---Executing _make_spinup_area_dfs()---
*** The following dict keys and their values have been added to 'scenario_dict ***
   'spinup_ha_input_df': Spinup input dataframe expressed per ha.
   'spinup_sko_input_df': Spinup input dataframe expressed per sko.
---Leaving _make_spinup_area_dfs()---
'spinup_ha_df' and 'spinup_sko_df' set using input_df for spinup_soc
>_make_spinup_area_dfs() completed
> 'h_value_dict' not set.
> Calling 'h_map_helper()'
---Executing h_map_helper()---
An h-value mapping dataframe does not exist.
Creating h_map_df from 'h_values.csv' in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/input
CIBUS crop mapping dataframe does not exist
> Calling 'crop_map_helper()'
---Executing crop_map_helper()---
No input dataframe exists.
Creating 'input_df' from 'crop_carbon_m

/home/niceri/Pythoncode/CIBUSmod_soil/notebooks/../CIBUSmod/soil/data_processing.py:680: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  spinup_sko_prodsys_sum_df = input_df.groupby(spinup_multi_groupby_idx).sum()


>spinup_df_to_soc_df finished<
>spinup_df_to_soc_df called to calculate SS SOC per sko
>spinup_df_to_soc_df finished<
>input_df_to_soc_df called to calculate SOC timeseries per ha
>input_df_to_soc_df finished<
>input_df_to_soc_df called to calculate SOC timeseries per sko
>input_df_to_soc_df finished<
Spinup SOC dataset saved as spinup_ha_soc_ds.nc 
 in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results
Spinup SOC dataset saved as spinup_sko_soc_ds.nc 
 in /home/niceri/Pythoncode/CIBUSmod_soil/data/soil/temp_results
*** The following dict keys and their values have been added to 'scenario_dict ***
   'historic_ha_soc_df': dataframe expressed per ha for historic SOC.
   'historic_sko_soc_df': dataframe expressed per sko for historic SOC.
   'historic_ha_soc_ds': xarray dataset of 'historic_ha_soc_df'.
   'historic_sko_soc_ds': xarray dataset of 'historic_sko_soc_df'.
---Leaving _calculate_historic_soc()---


In [1]:
spinup_ha_soc_ds = xr.load_dataset('intermediate_results/spinup_ha_soc_ds.nc')

NameError: name 'xr' is not defined

In [ ]:
ny_ds = spinup_ha_soc_ds+scn_ha_soc_ds

In [ ]:
scn_ha_soc_ds.tot_pool.isel(prod_system=0,region=0,fraction=0)

In [ ]:
scn_ha_soc_ds.tot_soc.sumisel(prod_system=0, region=0, fraction=0).plot()

In [ ]:
# Example usage

## The following functions recalculates the scenario and spinup dataframes per ha and per sko. 
## They both return two dataframes. The first expressed per ha, and the second per sko
#scn_ha_df, scn_sko_df = data_processing.make_scn_area_dfs(FAI_scn_df)
#spinup_ha_df, spinup_sko_df = data_processing._make_spinup_area_dfs(spinup_df)
## The following functions calculates the SOC timeseries based on the input dataframes
## They both return two dataframes. One per ha, and one per sko (same order as input)
#scn_ha_soc, scn_sko_soc = data_processing.calculate_scn_soc(scn_ha_df, scn_sko_df)
#spinup_ha_soc, spinup_sko_soc = data_processing._calculate_historic_soc(spinup_ha_df, spinup_sko_df)

# 3. Calculate and assign total SOC to the generated xarray datasets

In [ ]:
for i in scenario_dict.keys():
    print(i)

In [ ]:
il.reload(data_processing)
# Calculate the tot_soc vector in the soc datasets
scn_ha_soc_ds, scn_sko_soc_ds, spinup_ha_soc_ds, spinup_sko_soc_ds = [scenario_dict['scenario_ha_soc_ds'], scenario_dict['scenario_sko_soc_ds'], scenario_dict['spinup_ha_soc_ds'], scenario_dict['spinup_sko_soc_ds']]
ds_vector = [scn_ha_soc_ds, scn_sko_soc_ds, spinup_ha_soc_ds, spinup_sko_soc_ds]
data_processing.assign_tot_soc(ds_vector)
data_processing.assign_co2_flux(ds_vector)
scenario_dict['scenario_ha_soc_ds'] = scn_ha_soc_ds
scenario_dict['scenario_sko_soc_ds'] = scn_sko_soc_ds
scenario_dict['spinup_ha_soc_ds'] = spinup_ha_soc_ds
scenario_dict['spinup_sko_soc_ds'] = spinup_sko_soc_ds

In [ ]:
 df = scenario_dict['scenario_ha_soc_ds'].to_dataframe()

In [ ]:
scn_ha

In [ ]:
scenario_dict.keys()
a = scenario_dict['spinup_sko_soc_ds'].y_pool.isel(prod_system=0,region=0,fraction=0)
b = scenario_dict['spinup_sko_soc_ds'].o_pool.isel(prod_system=0,region=0,fraction=0)
c = a + b
c.plot()

In [ ]:
scenario_dict.keys()

In [ ]:
# Examples how to use the output from the functions in this notebook

In [ ]:
grouped_df = df.groupby(level=['prod_system', 'region', 'output_year']).sum()

# Iterate through unique values of 'prod_system'
for i in df.index.get_level_values('prod_system').unique():
    # Filter the DataFrame
    sub_df = grouped_df.loc[i, ['y_pool', 'o_pool', 'tot_soc', 'co2']]

    # Create subplots
    fig, axes = plt.subplots(nrows=len(sub_df.columns), figsize=(8, 10))
    
    # Iterate through each subplot
    for idx, (column, ax) in enumerate(zip(sub_df.columns, axes)):
        # Plot the data
        sub_df[column].plot(ax=ax, title=f'SKO: {i} - {column}')

        # Add a horizontal line at y=0
        ax.axhline(y=0, color='black', linestyle='--')

    # Adjust layout
    plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Show individual plots of the SOC evolution in all the SKO's execute this cell
## OBS! Many
#df = scenario_dict['scenario_ha_soc_df']
display.display(display.Markdown("## Plots of kg C/ha in Y- and O-pool in all sko's"))
for i in df.index.get_level_values('region').unique():
    df.groupby([ 'prod_system', 'region','output_year']).sum().loc[('conventional',i),('y_pool', 'o_pool', 'tot_soc', 'co2')].plot(title=f'SKO: {i}')

### Save/Read df, preserved
The below functions can be used to save and read back dataframes to/from csv files without loosing dtype and formatting

In [ ]:
# save the results in a csv file without loosing dtypes
soil_utils.to_csv_preserved(FAI_scn_df, save_as='FAI_scn_inputs_df', set_workdir='intermediate_results/')

In [ ]:
# load the saved csv file with original dtypes
il.reload(soil_utils)
scn_inputs_df = soil_utils.read_csv_preserved('intermediate_results/FAI_scn_inputs_df.csv')

# If changes to datasets are required - run this first. 
## Change mapping dataframes

The function make_h_map_df() calculates a multiindex dataframe mapping each crop in CIBUSmod to the above-, and below-ground fraction h-values, used in icbm.

* The function can be run without arguments. It then defaults to use preset csv-files.
* Alternatively, the 'h_map_df', 'crop_map_df' and 'amnd_map_df' can be constructed using 'run make_df' and executing the corresponding cell below.

It takes the crop_map_df, h_map_df and the amnd_map_df as inputs.

    Additional optional arguments
    re_col, map_col and output_col_name.
    These refer to the column names used when creating the crop_map_df (re_col_name='re');
    name of column used to map ammendments to crop type (map_col='h_value_type');
    and name of column name in the return dataframe (output_col_name='h_value')

In [ ]:
# Run this cell to create the h_value_map:

h_value_map_df = data_processing.make_h_map_df()

## function call examples:
### All default imports
#h_value_map_df = data_processing.make_h_map_df()
### h_map_value dataframe changed
#h_value_map_df = data_processing.make_h_map_df(h_map_df=h_map_df)
### All dataframes changed
#h_value_map_df = data_processing.make_h_map_df(h_map_df=h_map_df, crop_map_df=crop_map_df, amnd_map_df=amnd_map_df)

## Execute 'run make_df' to make changes to dataframes, or use new input files
The same call can be used for to change or generate any dataframe from a csv file

In [ ]:
run make_df

### Change mapping of h-values to different input sources (h_map_df)
* execute 'run make_df'
* Select 'h_values.csv' as input source, if unknown.
* Set 'h_value_type' and 'h_frac' as index columns
  
When the dataframe is ready, execute the cell below to create the 'h_map_df' variable

In [ ]:
# Create h_map_df
h_map_df = df_dict[list(df_dict)[0]]h

### Change  mappings between crops, c-allocations, re-values and h-value types (crop_map_df)
* execute 'run make_df'
* Select 'crop_carbon_map.csv' for default
* Set 'CIBUS' column to index

When the dataframe is ready, execute the cell below to create the 'crop_map_df' variable

In [ ]:
# Create crop_map_df
process_df = df_dict[list(df_dict)[0]]
crop_map_df, dict_crops = data_processing.crop_map_helper(process_df)

### Create a mapping of h-values to different amendment sources (amnd_map_df)
* execute 'run make_df'
* Select 'amnd_map.csv' as input source, if unknown.
* Set 'CIBUS' as index column

When the dataframe is ready, execute the cell below to create the 'amnd_map_df' variable

In [ ]:
# Create amnd_map_df
amnd_map_df = df_dict[list(df_dict)[0]]

In [ ]:
colored_rule(color='red', height=2)

## Execute 'run make_df' to make changes to dataframes, or use new input files
The same call can be used for to change or generate any dataframe from a csv file

In [ ]:
run make_df

# 3. Create dataframes with crops and C inputs

# THE CODE BELOW IS NOT OPERATIONAL

In [ ]:
class GeoUnit(traitlets.HasTraits):
    assigned_ids = set()
    unique_id = traitlets.Int() # unique id with no external significance
    sko_id = traitlets.Int() # Swedisk 'skördeområde'
    total_area = traitlets.Int() # Area in ha

    def __init__(self, sko_id):
        self.sko_id = sko_id
        self.unique_id = self.generate_unique_id()

    def generate_unique_id(self):
        new_id = len(self.assigned_ids) + 1
        while new_id in self.assigned_ids:
            new_id += 1

        self.assigned_ids.add(new_id)
        return new_id

colored_rule(color='red', height=2)